# Construindo um Tokenizador BPE e Explorando o WordPiece

Neste laboratório, implementamos os fundamentos do algoritmo Byte Pair Encoding (BPE) do zero e exploramos o funcionamento do WordPiece através da biblioteca Hugging Face Transformers.

## Tarefa 1: O Motor de Frequências

Implementaremos a função `get_stats(vocab)` que calcula as frequências de todos os pares adjacentes de caracteres/símbolos no vocabulário.

In [1]:
# Inicializar o vocabulário de treinamento (corpus)
vocab = {
    'l o w </w>': 5,
    'l o w e r </w>': 2,
    'n e w e s t </w>': 6,
    'w i d e s t </w>': 3
}

print("Vocabulário inicial (corpus de treinamento):")
for word, freq in vocab.items():
    print(f"  '{word}': {freq}")
print()

Vocabulário inicial (corpus de treinamento):
  'l o w </w>': 5
  'l o w e r </w>': 2
  'n e w e s t </w>': 6
  'w i d e s t </w>': 3



In [2]:
def get_stats(vocab):
    """
    Calcula as frequências de todos os pares adjacentes de caracteres/símbolos.
    
    Args:
        vocab (dict): Dicionário com palavras como chaves e frequências como valores.
                     As palavras devem estar separadas por espaços (ex: 'l o w </w>')
    
    Returns:
        dict: Dicionário com pares de símbolos como chaves e frequências como valores
    """
    pairs = {}
    
    for word, freq in vocab.items():
        # Separar a palavra em símbolos individuais
        symbols = word.split()
        
        # Contar frequência de pares adjacentes
        for i in range(len(symbols) - 1):
            pair = (symbols[i], symbols[i + 1])
            pairs[pair] = pairs.get(pair, 0) + freq
    
    return pairs

# Executar get_stats e exibir resultados
pairs_freq = get_stats(vocab)

print("Frequências de pares adjacentes:")
# Ordenar por frequência decrescente
for pair, count in sorted(pairs_freq.items(), key=lambda x: x[1], reverse=True):
    print(f"  {pair}: {count}")
print()

# Validação: o par ('e', 's') deve ter frequência 9
print("VALIDAÇÃO - Verificando o par ('e', 's'):")
pair_es = ('e', 's')
count_es = pairs_freq.get(pair_es, 0)
print(f"  Frequência do par {pair_es}: {count_es}")
print(f"  ✓ Validação passou!" if count_es == 9 else f"  ✗ Erro: esperado 9, obtido {count_es}")
print()

Frequências de pares adjacentes:
  ('e', 's'): 9
  ('s', 't'): 9
  ('t', '</w>'): 9
  ('w', 'e'): 8
  ('l', 'o'): 7
  ('o', 'w'): 7
  ('n', 'e'): 6
  ('e', 'w'): 6
  ('w', '</w>'): 5
  ('w', 'i'): 3
  ('i', 'd'): 3
  ('d', 'e'): 3
  ('e', 'r'): 2
  ('r', '</w>'): 2

VALIDAÇÃO - Verificando o par ('e', 's'):
  Frequência do par ('e', 's'): 9
  ✓ Validação passou!



## Tarefa 2: O Loop de Fusão (Merge)

Implementaremos a função `merge_vocab(pair, v_in)` que realiza a fusão do par mais frequente e o loop principal que executa 5 iterações do algoritmo BPE.

In [3]:
def merge_vocab(pair, v_in):
    """
    Realiza a fusão de um par específico no vocabulário.
    
    Substitui todas as ocorrências isoladas do par (a, b) pela versão unificada 'ab'
    em todas as palavras do vocabulário.
    
    Args:
        pair (tuple): Tuple contendo o par a ser fundido, ex: ('e', 's')
        v_in (dict): Dicionário de vocabulário atual
    
    Returns:
        dict: Novo dicionário com o par fundido
    """
    v_out = {}
    bigram = ' '.join(pair)
    replacement = ''.join(pair)
    
    for word, freq in v_in.items():
        # Substituir o par pela versão unificada
        new_word = word.replace(bigram, replacement)
        v_out[new_word] = freq
    
    return v_out

print("Função merge_vocab definida com sucesso!")
print()

Função merge_vocab definida com sucesso!



In [4]:
# Loop Principal de Treinamento do Tokenizador BPE (K=5 iterações)
print("=" * 70)
print("LOOP PRINCIPAL DE TREINAMENTO BPE (5 Iterações)")
print("=" * 70)
print()

# Reinicializar vocab para o loop
vocab = {
    'l o w </w>': 5,
    'l o w e r </w>': 2,
    'n e w e s t </w>': 6,
    'w i d e s t </w>': 3
}

num_iterations = 5

for i in range(num_iterations):
    print(f"--- Iteração {i + 1} ---")
    
    # Obter as frequências dos pares
    pairs = get_stats(vocab)
    
    if not pairs:
        print("Nenhum par encontrado. Finalizando.")
        break
    
    # Encontrar o par mais frequente
    best = max(pairs, key=pairs.get)
    freq_best = pairs[best]
    
    print(f"Par mais frequente: {best} com frequência {freq_best}")
    
    # Realizar a fusão
    vocab = merge_vocab(best, vocab)
    
    # Exibir o estado do vocabulário após a fusão
    print(f"Vocabulário após fusão:")
    for word, freq in vocab.items():
        print(f"  '{word}': {freq}")
    
    print()

print("=" * 70)
print("OBSERVAÇÕES FINAIS")
print("=" * 70)
print("Após as 5 iterações, observe a formação de tokens morfológicos:")
print("  - Sufixos como 'est</w>' e 'er</w>'")
print("  - Fusões progressivas de caracteres comuns")
print()

LOOP PRINCIPAL DE TREINAMENTO BPE (5 Iterações)

--- Iteração 1 ---
Par mais frequente: ('e', 's') com frequência 9
Vocabulário após fusão:
  'l o w </w>': 5
  'l o w e r </w>': 2
  'n e w es t </w>': 6
  'w i d es t </w>': 3

--- Iteração 2 ---
Par mais frequente: ('es', 't') com frequência 9
Vocabulário após fusão:
  'l o w </w>': 5
  'l o w e r </w>': 2
  'n e w est </w>': 6
  'w i d est </w>': 3

--- Iteração 3 ---
Par mais frequente: ('est', '</w>') com frequência 9
Vocabulário após fusão:
  'l o w </w>': 5
  'l o w e r </w>': 2
  'n e w est</w>': 6
  'w i d est</w>': 3

--- Iteração 4 ---
Par mais frequente: ('l', 'o') com frequência 7
Vocabulário após fusão:
  'lo w </w>': 5
  'lo w e r </w>': 2
  'n e w est</w>': 6
  'w i d est</w>': 3

--- Iteração 5 ---
Par mais frequente: ('lo', 'w') com frequência 7
Vocabulário após fusão:
  'low </w>': 5
  'low e r </w>': 2
  'n e w est</w>': 6
  'w i d est</w>': 3

OBSERVAÇÕES FINAIS
Após as 5 iterações, observe a formação de tokens morfo

## Tarefa 3: Integração Industrial - WordPiece e Hugging Face

Agora exploraremos o funcionamento do WordPiece, que é utilizado em modelos como BERT. O WordPiece opera com regras probabilísticas ligeiramente diferentes do BPE clássico, priorizando a fusão de pares que melhoram a probabilidade geral do corpus.

In [5]:
# Instalar biblioteca transformers (descomente se necessário)
# !pip install transformers

# Importar o tokenizador multilíngue do BERT
from transformers import AutoTokenizer

print("Instalando e carregando o tokenizador multilíngue do BERT...")
print("(Isso pode levar alguns minutos na primeira execução)")
print()

# Instanciar o tokenizador multilíngue
tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")

print("✓ Tokenizador carregado com sucesso!")
print(f"Tipo: {type(tokenizer)}")
print()

c:\Users\Usuario\Desktop\ELetiva2\Construindo-um-Tokenizador-BPE-e-Explorando-o-WordPiece\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Instalando e carregando o tokenizador multilíngue do BERT...
(Isso pode levar alguns minutos na primeira execução)

✓ Tokenizador carregado com sucesso!
Tipo: <class 'transformers.models.bert.tokenization_bert.BertTokenizer'>



In [6]:
# Frase de teste em português (projetada para forçar particionamento morfológico)
test_sentence = "Os hiper-parâmetros do transformer são inconstitucionalmente difíceis de ajustar."

print("=" * 70)
print("TESTE DE TOKENIZAÇÃO WORDPIECE")
print("=" * 70)
print()
print(f"Frase original:")
print(f"  \"{test_sentence}\"")
print()

# Tokenizar a frase
tokens = tokenizer.tokenize(test_sentence)

print(f"Tokens gerados (total: {len(tokens)}):")
print()
for i, token in enumerate(tokens, 1):
    print(f"  {i:2d}. {token:20s}", end="")
    if i % 3 == 0:
        print()
    else:
        print(" | ", end="")

print("\n")
print("=" * 70)
print()

TESTE DE TOKENIZAÇÃO WORDPIECE

Frase original:
  "Os hiper-parâmetros do transformer são inconstitucionalmente difíceis de ajustar."

Tokens gerados (total: 27):

   1. Os                   |    2. hip                  |    3. ##er                
   4. -                    |    5. par                  |    6. ##âm                
   7. ##etros              |    8. do                   |    9. transform           
  10. ##er                 |   11. são                  |   12. in                  
  13. ##cons               |   14. ##tit                |   15. ##uc                
  16. ##ional              |   17. ##mente              |   18. di                  
  19. ##f                  |   20. ##í                  |   21. ##cei               
  22. ##s                  |   23. de                   |   24. aj                  
  25. ##usta               |   26. ##r                  |   27. .                   





### Análise dos Tokens WordPiece

## Significado dos Símbolos ## (Hash duplo)

Nos tokens acima, você pode observar símbolos `##` precedendo alguns tokens (ex: `##mente`, `##mente`, `##dor`). Estes símbolos indicam que o token é uma **continuação de uma palavra anterior**, não o início de uma nova palavra.

### Como funciona:

1. **Tokens sem ##**: Representam o início de uma palavra ou uma palavra completa
   - Exemplo: `Os`, `hiper`, `do`

2. **Tokens com ##**: Representam partes subsequentes de uma palavra
   - Exemplo: `##mente` significa que faz parte de uma palavra que começou anteriormente
   - A palavra `inconstitucionalmente` é decomposta em: `incons` + `##tituc` + `##ion` + `##al` + `##mente`

### Por que isso previne falhas com vocabulário desconhecido:

Se o modelo encontrar uma palavra completamente desconhecida, ele pode:
1. Decompô-la em sub-palavras conhecidas
2. Cada sub-palavra tem sua própria incorporação (embedding)
3. O modelo processa essas sub-palavras através da rede neural Transformer
4. A contexto da frase ajuda a inferir o significado mesmo se a palavra completa for nova

Isso torna o modelo **muito mais robusto** contra palavras raras, escrita incorreta e vocabulário fora do domínio de treinamento.